## Part 4 of 6: Feature Engineering

Loads df from notebooks/03_eda.ipynb.

See `notebooks/README.md` for the full run order.

In [ ]:
# Import Python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries have been imported!")

In [ ]:
# Load state saved by the previous notebook
import joblib
_state = joblib.load("_state/03_state.joblib")
globals().update(_state)
print(f"Loaded {len(_state)} objects: {sorted(_state)}")


In [ ]:
# Feature engineering: Ratios, trends, Z-Scores
# All feature engineering occurs before train/val/test split
# Anything that is fit only on training is fit on the TRAIN_MASK

# Build training mask
TRAIN_MASK = df["year"] <= 2011

# Build financial ratios
# If a denominator is 0 or close to 0, return NaN and impute later on
def safe_div(num, den, floor=1e-6):
    """Ratio with near-zero/negative denominators set to NaN."""
    out = num / den.where(den.abs() >= floor)
    return out.replace([np.inf, -np.inf], np.nan)

# Flag firms that have no inventory, so they don't get imputed
# This is actually 25% of the data!
df["inventory_is_zero"] = (df["inventory"] == 0).astype(int)
print(f"Rows with zero inventory: {df['inventory_is_zero'].sum():,} "
      f"({df['inventory_is_zero'].mean():.1%})")

# This is what we've all been waiting for
# Build the Altman Z-Score
df["altman_x1"] = safe_div(df["current_assets"] - df["curr_liabilities"], df["total_assets"])
df["altman_x2"] = safe_div(df["retained_earnings"], df["total_assets"])
df["altman_x3"] = safe_div(df["ebit"], df["total_assets"])
df["altman_x4"] = safe_div(df["market_value"], df["total_liabilities"])
df["altman_x5"] = safe_div(df["net_sales"], df["total_assets"])
df["altman_z"] = (1.2 * df["altman_x1"] + 1.4 * df["altman_x2"] +
                  3.3 * df["altman_x3"] + 0.6 * df["altman_x4"] +
                  1.0 * df["altman_x5"])

# Profitability metrics
df["roa"]          = safe_div(df["net_income"], df["total_assets"])
df["ebitda_ta"]    = safe_div(df["ebitda"], df["total_assets"])
df["net_margin"]   = safe_div(df["net_income"], df["total_revenue"].where(df["total_revenue"] > 0))
df["gross_margin"] = safe_div(df["gross_profit"], df["net_sales"].where(df["net_sales"] > 0))
# Leverage metrics
df["liab_ta"]   = safe_div(df["total_liabilities"], df["total_assets"])
df["ltdebt_ta"] = safe_div(df["lt_debt"], df["total_assets"])
# Liquidity & efficiency metrics
df["current_ratio"]      = safe_div(df["current_assets"], df["curr_liabilities"])
df["inventory_turnover"] = safe_div(df["cogs"], df["inventory"])
df["dso"]                = safe_div(df["receivables"], df["net_sales"].where(df["net_sales"] > 0)) * 365

# Remember the year gaps from earlier?
# Let's average out the changes so we have a picture of change per year
g = df.groupby("company_name")
gap1 = g["year"].diff().eq(1)
gap2 = gap1 & gap1.groupby(df["company_name"]).shift(1).fillna(False)

# Gap changes in each variable
df["d_ebitda_ta"] = (g["ebitda"].diff() / df["total_assets"]).where(gap1)
df["d_net_income_ta"] = (g["net_income"].diff() / df["total_assets"]).where(gap1)
df["pct_chg_ca"] = (g["current_assets"].pct_change().where(gap1).replace([np.inf, -np.inf], np.nan))
df["ebitda_slope2_ta"] = ((df["ebitda"] - g["ebitda"].shift(2)) / 2 / df["total_assets"]).where(gap2)
df["d_inv_turnover"] = g["inventory_turnover"].diff().where(gap1)

# Consecutive years of negative net income
neg = (df["net_income"] < 0).astype(int)
block = ((neg != neg.shift()) | (df["company_name"] != df["company_name"].shift())).cumsum()
df["consec_neg_ni"] = neg.groupby(block).cumsum()

# Any undefined trends are imputed as 0, plus a history counter is added
TREND_COLS = ["d_ebitda_ta", "d_net_income_ta", "pct_chg_ca",
              "ebitda_slope2_ta", "d_inv_turnover"]
df[TREND_COLS] = df[TREND_COLS].fillna(0)
df["years_of_history"] = g.cumcount() + 1

# Number of NaN counts
RATIO_COLS = ["altman_x1", "altman_x2", "altman_x3", "altman_x4", "altman_x5",
              "roa", "ebitda_ta", "net_margin", "gross_margin",
              "liab_ta", "ltdebt_ta", "current_ratio", "inventory_turnover",
              "dso", "altman_z"]
print("NaN counts before imputation (zero/negative denominators):")
print(df[RATIO_COLS].isnull().sum().loc[lambda s: s > 0].to_string())

In [ ]:
# Continue the cleaning: Winsorize, impute, and transform data

# Winsorize engineered ratios and trend features at train-window 1st/99th pct
CAP_COLS = RATIO_COLS + TREND_COLS
caps = df.loc[TRAIN_MASK, CAP_COLS].quantile([0.01, 0.99])
df[CAP_COLS] = df[CAP_COLS].clip(lower=caps.loc[0.01], upper=caps.loc[0.99], axis=1)
print("Winsorized ratios/trends at train-derived 1st/99th pct.")

# Impute all remaining NaNs with TRAIN medians only
medians = df.loc[TRAIN_MASK, RATIO_COLS].median()
df[RATIO_COLS] = df[RATIO_COLS].fillna(medians)

# Size control: Skew check, train-cap winsorize, then log1p.
print(f"total_assets skew before: {df['total_assets'].skew():.1f}")
ta_caps = df.loc[TRAIN_MASK, "total_assets"].quantile([0.01, 0.99])
df["log_total_assets"] = np.log1p(df["total_assets"].clip(ta_caps[0.01], ta_caps[0.99]))
print(f"log_total_assets skew after: {df['log_total_assets'].skew():.2f}")

# Ratio correlation heatmap
corr_ratio = df[RATIO_COLS].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_ratio, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7})
plt.title("Engineered ratios: Pearson correlation")
plt.tight_layout(); plt.show()
pairs = [(a, b, corr_ratio.loc[a, b]) for i, a in enumerate(RATIO_COLS)
         for b in RATIO_COLS[i + 1:] if abs(corr_ratio.loc[a, b]) > 0.85]
print("Ratio pairs above r =|0.85| that may risk multicollinearity:")
for a, b, c in pairs:
    print(f"  {a} vs {b}: {c:+.2f}")
print("Note: altman_z correlating with its own components is likely expected.")

In [ ]:
# K-Means unsupervised clustering as a step to further prepare the data for modeling.
# Testing whether unsupervised structure on the engineered ratios recovers segmentation.
# Fit on training split only, then applied to every row.
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Exclude altman_z: it's a fixed linear combination of altman_x1-x5, which are
# already in RATIO_COLS, so including it would cause multicollinearity.
CLUSTER_COLS = [c for c in RATIO_COLS if c != "altman_z"]
clust_scaler = StandardScaler().fit(df.loc[TRAIN_MASK, CLUSTER_COLS])
Xc_train = clust_scaler.transform(df.loc[TRAIN_MASK, CLUSTER_COLS])
Xc_all = clust_scaler.transform(df[CLUSTER_COLS])

# Elbow method and silhouette score by k
# Note: Silhouette is O(n^2), which is slow, so it's computed on a fixed subsample.
K_RANGE = range(2, 11)
inertias, sils = [], []
for k in K_RANGE:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xc_train)
    inertias.append(km_k.inertia_)
    sils.append(silhouette_score(Xc_train, km_k.labels_, sample_size=5000, random_state=42))

# Plot elbow method and silhouette score
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow method")
axes[1].plot(list(K_RANGE), sils, marker="o", color="#C44E52")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette by k")
plt.tight_layout(); plt.show()

# Select a number of clusters
K_OPT = list(K_RANGE)[int(np.argmax(sils))]
print(f"Silhouette-selected k = {K_OPT} (silhouette={max(sils):.3f}). ")
kmeans = KMeans(n_clusters=K_OPT, n_init=10, random_state=42).fit(Xc_train)
df["industry_cluster"] = kmeans.predict(Xc_all)

# Print out clusters
print(df.groupby("industry_cluster")["target"].agg(["mean", "size"]).round(4).to_string())

# One-hot encode and append as candidate model features
# Hopefully this improves model AUC later down the line
cluster_dummies = pd.get_dummies(df["industry_cluster"], prefix="industry_cluster").astype(int)
CLUSTER_FEATURE_COLS = cluster_dummies.columns.tolist()
df[CLUSTER_FEATURE_COLS] = cluster_dummies
print(f"Added {len(CLUSTER_FEATURE_COLS)} one-hot cluster columns to df.")

In [ ]:
# Define feature sets for different model types
FEATURES_ALL = (RATIO_COLS + TREND_COLS +
                ["consec_neg_ni", "years_of_history", "inventory_is_zero",
                 "log_total_assets"] + ["industry_cluster_1"])

# Feature set for Logistic Regression
# Remove industry cluster and altman_z due to potential multicollinearity
FEATURES_LR = [f for f in FEATURES_ALL if f not in ["industry_cluster_1", "altman_z"]]

# Feature set for Decision Tree
# Remove altman_z due to potential overfitting on Altman Z-Score splits
FEATURES_DT = [f for f in FEATURES_ALL if f != "altman_z"]

# Standard FEATURES pointer for tree models
FEATURES = FEATURES_ALL

# Year is used only here, for splitting.
tr = df[df["year"].between(1999, 2011)]
va = df[df["year"].between(2012, 2014)]
te = df[df["year"].between(2015, 2018)]

# Prepare generic splits for tree models
X_tr, y_tr = tr[FEATURES], tr["target"]
X_va, y_va = va[FEATURES], va["target"]
X_te, y_te = te[FEATURES], te["target"]

# Prepare specific splits for Logistic Regression
X_tr_lr, X_va_lr, X_te_lr = tr[FEATURES_LR], va[FEATURES_LR], te[FEATURES_LR]
print(f"Training set: {X_tr.shape[0]} rows")
print(f"LR features ({len(FEATURES_LR)}): {FEATURES_LR}")
print(f"Tree features ({len(FEATURES_ALL)}): {FEATURES_ALL}")

# Prepare specific splits for Decision Tree
X_tr_dt, X_va_dt, X_te_dt = tr[FEATURES_DT], va[FEATURES_DT], te[FEATURES_DT]

# Function to build evaluation of models
def evaluate(name, y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {"model": name,
            "ROC-AUC": roc_auc_score(y_true, scores),
            "PR-AUC": average_precision_score(y_true, scores),
            "F1 (failed)": f1_score(y_true, pred),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred),
            "threshold": threshold}

# Function to determine F1 threshhold
def best_f1_threshold(y_true, scores):
    p, r, t = precision_recall_curve(y_true, scores)
    f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
    return float(t[np.argmax(f1[:-1])])

In [ ]:
# Save state for the next notebook
import joblib
from pathlib import Path
Path("_state").mkdir(exist_ok=True)
joblib.dump({
    "df": df, "FEATURES": FEATURES, "FEATURES_LR": FEATURES_LR, "FEATURES_DT": FEATURES_DT, "te": te, "X_tr": X_tr, "y_tr": y_tr, "X_va": X_va, "y_va": y_va, "X_te": X_te, "y_te": y_te, "X_tr_lr": X_tr_lr, "X_va_lr": X_va_lr, "X_te_lr": X_te_lr, "X_tr_dt": X_tr_dt, "X_va_dt": X_va_dt, "X_te_dt": X_te_dt
}, "_state/04_state.joblib")
